In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from operator import add
from time import sleep

class OverAllState(TypedDict):
    logs: Annotated[list[str], add]
    # 如果出现并行节点同时更新状态往下游传递的时候，必须要有reducer
    id: str

def node_a(state: OverAllState):
    for k,v in state.items():
        print(f"ka: {k}, v: {v}")
    return {
        "logs": ["node_a 更新状态"]
    }

def node_b(state: OverAllState):
    sleep(2)
    for k,v in state.items():
        print(f"kb: {k}, v: {v}")
    return {
        "logs": ["node_b 更新状态"]
    }

def node_c(state: OverAllState):
    for k,v in state.items():
        print(f"kc: {k}, v: {v}")
    return {
        "logs": ["node_c 更新状态"]
    }
    

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)
builder.add_node("node_c", node_c)

builder.add_edge(START, "node_a")
builder.add_edge(START, "node_b")
builder.add_edge(START, "node_c")

builder.add_edge("node_a", END)
builder.add_edge("node_b", END)
builder.add_edge("node_c", END)

graph = builder.compile()
result = graph.invoke({"logs": ["START"], "id": "start"})
print('=' * 30, '-> result <-', '=' * 30)
print(result)

ka: logs, v: ['START']
ka: id, v: start
kc: logs, v: ['START']
kc: id, v: start
kb: logs, v: ['START']
kb: id, v: start
============================== -> result <- ==============================
{'logs': ['START', 'node_a 更新状态', 'node_b 更新状态', 'node_c 更新状态'], 'id': 'start'}
